# 4장 모델 평가와 하이퍼파라미터 고르기 — 검증, 교차 검증, 혼동 행렬, 정밀도·재현율, 그리드 서치
『머신러닝과 딥러닝 기초』 실습 노트북

다음을 해 봅니다.
1. 검증 데이터로 k-최근접 이웃의 k 고르기 (펭귄 성별)
2. 교차 검증으로 더 믿을 만하게 k 고르기
3. 유방암 진단 데이터로 혼동 행렬, 정밀도, 재현율 계산하기
4. 결정 임곗값을 바꾸어 정밀도와 재현율 조절하기
5. GridSearchCV로 하이퍼파라미터 한꺼번에 고르기

위에서부터 차례로 실행하세요.

## 1. 검증 데이터로 k 고르기
3장처럼 측정값 네 개로 펭귄의 성별을 맞힙니다. 학습 데이터 249마리, 테스트 데이터 84마리로 나눕니다.

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

df = sns.load_dataset("penguins")
num = ["bill_length_mm", "bill_depth_mm",
       "flipper_length_mm", "body_mass_g"]
data = df.dropna(subset=["sex"])
X, y = data[num], data["sex"]
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

학습 데이터를 다시 학습용 186마리와 검증용 63마리로 나누고, k를 1, 3, 5, …, 31로 바꾸어 검증 정확도가 가장 높은 k를 찾습니다.

In [ ]:
# 학습 데이터를 다시 학습용과 검증용으로 나눈다
Xa, Xv, ya, yv = train_test_split(
    Xtr, ytr, test_size=0.25, random_state=0, stratify=ytr)

scores = {}
for k in range(1, 32, 2):
    model = make_pipeline(StandardScaler(),
                          KNeighborsClassifier(n_neighbors=k))
    model.fit(Xa, ya)                     # 학습용 186마리로 학습
    scores[k] = model.score(Xv, yv)       # 검증용 63마리로 평가

# 검증 정확도가 가장 높은 k (같은 점수가 여럿이면 앞의 작은 k)
k_best = max(scores, key=scores.get)
print(f"고른 k = {k_best}, 검증 정확도 {scores[k_best]:.1%}")

검증 데이터를 나누는 `random_state`만 바꾸면 고른 k가 어떻게 달라질까요?

In [ ]:
for rs in [1, 2, 3]:
    Xa, Xv, ya, yv = train_test_split(
        Xtr, ytr, test_size=0.25, random_state=rs, stratify=ytr)
    scores = {}
    for k in range(1, 32, 2):
        model = make_pipeline(StandardScaler(),
                              KNeighborsClassifier(n_neighbors=k))
        scores[k] = model.fit(Xa, ya).score(Xv, yv)
    k_best = max(scores, key=scores.get)
    print(f"random_state={rs}: 고른 k = {k_best}, "
          f"검증 정확도 {scores[k_best]:.1%}")

## 2. 교차 검증
5-폴드 교차 검증으로 k = 5 모델을 평가합니다. 폴드마다 정확도가 얼마나 다른지 보세요.

In [ ]:
from sklearn.model_selection import cross_val_score

model = make_pipeline(StandardScaler(),
                      KNeighborsClassifier(n_neighbors=5))
cv = cross_val_score(model, Xtr, ytr, cv=5)    # 5-폴드
print("폴드별 정확도:", cv.round(3))
print(f"평균 {cv.mean():.1%}, 표준편차 {cv.std():.1%}")

k를 바꾸어 가며 교차 검증 평균이 가장 높은 k를 고릅니다 (80번 학습하므로 몇 초 걸립니다).

In [ ]:
cv_mean = {}
for k in range(1, 32, 2):
    model = make_pipeline(StandardScaler(),
                          KNeighborsClassifier(n_neighbors=k))
    cv_mean[k] = cross_val_score(model, Xtr, ytr, cv=5).mean()

k_best = max(cv_mean, key=cv_mean.get)
print(f"고른 k = {k_best}, 교차 검증 정확도 {cv_mean[k_best]:.1%}")

고른 k로 학습 데이터 전체를 써서 다시 학습하고, 테스트 데이터로 **한 번** 평가합니다.

In [ ]:
model = make_pipeline(StandardScaler(),
                      KNeighborsClassifier(n_neighbors=9))
model.fit(Xtr, ytr)
print(f"테스트 정확도: {model.score(Xte, yte):.1%}")

## 3. 유방암 진단 데이터
세포핵의 크기·모양·질감을 잰 30가지 수치로 혹이 악성인지 맞힙니다. 찾고 싶은 쪽(악성)을 1로 바꿉니다.

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)
X = cancer.data                           # 특징 30개
y = (cancer.target == 0).astype(int)      # 악성이면 1, 아니면 0
print(X.shape)
print(y.value_counts())

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(),
                      KNeighborsClassifier(n_neighbors=5))
model.fit(Xtr, ytr)
print(f"테스트 정확도: {model.score(Xte, yte):.1%}")

## 4. 혼동 행렬
행은 실제 정답, 열은 예측입니다. `[[TN, FP], [FN, TP]]` 순서입니다.

In [ ]:
from sklearn.metrics import confusion_matrix

pred = model.predict(Xte)
print(confusion_matrix(yte, pred))

## 5. 정밀도, 재현율, F1 점수
- 정밀도 = TP / (TP + FP): 악성이라고 한 것 가운데 진짜 악성
- 재현율 = TP / (TP + FN): 진짜 악성 가운데 찾아낸 것

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"정밀도 {precision_score(yte, pred):.1%}")
print(f"재현율 {recall_score(yte, pred):.1%}")
print(f"F1 점수 {f1_score(yte, pred):.1%}")

## 6. 결정 임곗값
이웃 5개 가운데 악성이 몇 개 이상이면 악성이라고 답할지 바꾸어 봅니다.

In [ ]:
proba = model.predict_proba(Xte)[:, 1]    # 악성 이웃의 비율
for n in [1, 2, 3, 4, 5]:     # 악성 이웃이 n개 이상이면 악성
    p = (proba >= n / 5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, p).ravel()
    print(f"{n}개 이상: 놓친 암 {fn}, 잘못된 경보 {fp}, "
          f"정밀도 {precision_score(yte, p):.1%}, "
          f"재현율 {recall_score(yte, p):.1%}")

## 7. 그리드 서치
k, 거리(유클리드/맨해튼), 이웃의 무게(같게/가까울수록 크게)의 44가지 조합을 5-폴드 교차 검증으로 비교합니다. F1 점수를 기준으로 고릅니다.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
# 이름 = 파이프라인 단계 이름 + "__" + 하이퍼파라미터 이름
grid = {
    "kneighborsclassifier__n_neighbors": list(range(1, 22, 2)),
    "kneighborsclassifier__metric": ["euclidean", "manhattan"],
    "kneighborsclassifier__weights": ["uniform", "distance"],
}
search = GridSearchCV(pipe, grid, cv=5, scoring="f1")
search.fit(Xtr, ytr)    # 44가지 x 5폴드 = 220번 + 다시 학습 1번

for name, value in search.best_params_.items():
    print(name.split("__")[1], "=", value)
print(f"교차 검증 F1 점수: {search.best_score_:.1%}")

가장 좋은 조합으로 다시 학습된 모델을 테스트 데이터로 한 번 평가합니다.

In [ ]:
pred = search.predict(Xte)
print(confusion_matrix(yte, pred))
print(f"정밀도 {precision_score(yte, pred):.1%}, "
      f"재현율 {recall_score(yte, pred):.1%}")

---
## 실습 문제

**문제 1.** 2장의 종 분류(특징: 부리 길이, 날개 길이)에서 5-폴드 교차 검증으로 k를 골라 보세요(1\~31 홀수, `StandardScaler`와 파이프라인 사용). 데이터는 2장처럼 `df.dropna(subset=["bill_length_mm", "flipper_length_mm"])`에서 종(`species`)을 레이블로 하고, `test_size=0.25, random_state=42, stratify=y`로 나누세요. 고른 k의 테스트 정확도는 얼마인가요?

In [ ]:
# 문제 1 풀이

**문제 2.** 이 노트북 2절(이북 4.3절)의 `cross_val_score(model, Xtr, ytr, cv=5)`에서 `cv=5` 대신 `StratifiedKFold(5, shuffle=True, random_state=rs)`를 넘겨, `rs`를 0, 1, 2로 바꾸어 평균과 표준편차를 비교해 보세요(`from sklearn.model_selection import StratifiedKFold`). 이 노트북 1절(이북 4.2절)의 홀드아웃 검증과 비교하면 결과가 얼마나 안정적인가요?

In [ ]:
# 문제 2 풀이

**문제 3.** 이 노트북 7절(이북 4.7절)의 그리드 서치에서 `scoring="f1"`을 `scoring="recall"`로 바꾸어 보세요. 어떤 조합이 골라지고, 테스트 데이터의 정밀도와 재현율은 어떻게 되나요? F1 점수로 고른 결과와 비교해 설명하세요.

In [ ]:
# 문제 3 풀이

**문제 4.** 의학에서는 재현율(민감도)과 함께 **특이도** = TN / (TN + FP)를 봅니다. 이 노트북 3절의 k = 5 모델과 7절의 그리드 서치 모델의 특이도를 각각 구해 보세요.

In [ ]:
# 문제 4 풀이